# Arrays in Python — Dynamic Arrays, Memory Layout & Algorithmic Patterns

> **Topic:** Arrays & Sequences | **Folder:** Data Structures & Algorithms

An **Array** is a contiguous block of memory holding a collection of elements.
In Python, arrays are implemented dynamically as built-in `list` objects, or as typed numerical arrays using the built-in `array` module.

---

## Table of Contents
1. [Memory Layout & Array Concepts](#1.-Memory-Layout-&-Array-Concepts)
2. [Python's Typed `array` Module](#2.-Python's-Typed-`array`-Module)
3. [Python `list` Under the Hood (Dynamic Resizing & Over-allocation)](#3.-Python-`list`-Under-the-Hood-(Dynamic-Resizing-&-Over-allocation))
4. [Time & Space Complexity of Array Operations](#4.-Time-&-Space-Complexity-of-Array-Operations)
5. [2D Arrays & Matrices (Representation & Shallow Copy Pitfalls)](#5.-2D-Arrays-&-Matrices-(Representation-&-Shallow-Copy-Pitfalls))
6. [Algorithmic Pattern 1: Two Pointers Technique](#6.-Algorithmic-Pattern-1:-Two-Pointers-Technique)
7. [Algorithmic Pattern 2: Sliding Window Technique](#7.-Algorithmic-Pattern-2:-Sliding-Window-Technique)
8. [Algorithmic Pattern 3: Prefix Sum Array](#8.-Algorithmic-Pattern-3:-Prefix-Sum-Array)
9. [Algorithmic Pattern 4: Kadane's Algorithm (Max Subarray Sum)](#9.-Algorithmic-Pattern-4:-Kadane's-Algorithm-(Max-Subarray-Sum))
10. [Quick Reference Card](#10.-Quick-Reference-Card)


---
## 1. Memory Layout & Array Concepts

- **Static Array**: Fixed-size contiguous block of memory. Index lookup formula:  
  $$\text{Address}(i) = \text{BaseAddress} + (i \times \text{ElementSize})$$
  Index lookup is $O(1)$ because memory offsets are calculated instantly in hardware.
- **Dynamic Array**: Automatically resizes (grows/shrinks) when capacity limits are reached.


In [ ]:
# Array index lookup demo
arr = [10, 20, 30, 40, 50]

print(f"Element at index 0: {arr[0]}")
print(f"Element at index 3: {arr[3]}")
print(f"Last element      : {arr[-1]}")


---
## 2. Python's Typed `array` Module

Python's standard library `array` module provides C-style typed arrays.
Unlike Python lists (which store pointers to full objects), `array.array` stores raw contiguous bytes.

| Type Code | C Type | Python Type | Size (Bytes) |
|-----------|--------|-------------|--------------|
| `'b'` / `'B'` | signed/unsigned char | int | 1 |
| `'i'` / `'I'` | signed/unsigned int | int | 2 or 4 |
| `'f'` / `'d'` | float / double | float | 4 / 8 |


In [ ]:
import array
import sys

# Create a typed array of signed integers ('i')
typed_arr = array.array('i', [10, 20, 30, 40, 50])
py_list   = [10, 20, 30, 40, 50]

print("Typed Array:", typed_arr)
print(f"Memory size typed array: {sys.getsizeof(typed_arr)} bytes")
print(f"Memory size Python list: {sys.getsizeof(py_list)} bytes")


---
## 3. Python `list` Under the Hood (Dynamic Resizing)

A Python `list` is a **dynamic array of pointers**.
When elements are added beyond capacity, CPython over-allocates memory using the formula:
$$\text{new\_capacity} = \text{new\_size} + (\text{new\_size} \gg 3) + (3 \text{ if new\_size < 9 else } 6)$$
This guarantees $O(1)$ **amortized** time complexity for `.append()`.


In [ ]:
# Inspecting dynamic resizing of Python lists using sys.getsizeof()
import sys

lst = []
old_size = sys.getsizeof(lst)
print(f"Initial empty list size: {old_size} bytes")

growth_log = []
for i in range(25):
    lst.append(i)
    current_size = sys.getsizeof(lst)
    if current_size != old_size:
        growth_log.append((len(lst), current_size))
        old_size = current_size

print("\nList Resizing Triggers (Length, Memory Bytes):")
for length, mem in growth_log:
    print(f"  Length: {length:>2} items -> Memory: {mem} bytes")


---
## 4. Time & Space Complexity of Array Operations

| Operation | Time Complexity | Notes |
|-----------|-----------------|-------|
| Index Lookup `arr[i]` | $O(1)$ | Direct pointer arithmetic |
| Append `arr.append(x)` | $O(1)$ Amortized | Occasional $O(n)$ memory allocation |
| Pop End `arr.pop()` | $O(1)$ | No element shifting required |
| Insert `arr.insert(i, x)` | $O(n)$ | Must shift elements right |
| Delete `arr.pop(i)` / `del` | $O(n)$ | Must shift elements left |
| Search `x in arr` | $O(n)$ | Linear scan |


---
## 5. 2D Arrays & Matrices

### Shallow Copy Pitfall
`[[0] * cols] * rows` creates `rows` references to the **SAME inner list**!  
Updating one row mutates all rows.

**Correct 2D matrix instantiation**:
`matrix = [[0 for _ in range(cols)] for _ in range(rows)]`


In [ ]:
# Shallow copy trap demonstration
rows, cols = 3, 3
bad_matrix = [[0] * cols] * rows
bad_matrix[0][0] = 99
print("Bad Matrix (all rows mutated!):", bad_matrix)

# Correct Matrix Instantiation
good_matrix = [[0 for _ in range(cols)] for _ in range(rows)]
good_matrix[0][0] = 99
print("Good Matrix (only 1 cell mutated):", good_matrix)


---
## 6. Algorithmic Pattern 1: Two Pointers Technique

Uses two indices (pointers) moving toward each other or in the same direction.
- **Time Complexity**: $O(n)$
- **Space Complexity**: $O(1)$


In [ ]:
# Two Pointers: In-place Array Reversal
def reverse_array_in_place(arr):
    left, right = 0, len(arr) - 1
    while left < right:
        arr[left], arr[right] = arr[right], arr[left]
        left += 1
        right -= 1
    return arr

data = [1, 2, 3, 4, 5, 6]
print("Original:", data)
print("Reversed:", reverse_array_in_place(data))


---
## 7. Algorithmic Pattern 2: Sliding Window Technique

Maintains a running contiguous window of elements to avoid recalculating overlapping subproblems.
- **Time Complexity**: $O(n)$ instead of $O(n \times k)$


In [ ]:
# Max sum subarray of size k
def max_sub_array_of_size_k(k, arr):
    if len(arr) < k: return 0
    
    window_sum = sum(arr[:k])
    max_sum = window_sum
    
    for window_end in range(k, len(arr)):
        window_sum += arr[window_end] - arr[window_end - k]  # Add incoming, subtract outgoing
        max_sum = max(max_sum, window_sum)
        
    return max_sum

nums = [2, 1, 5, 1, 3, 2]
k = 3
print(f"Max sum of subarray of size {k}:", max_sub_array_of_size_k(k, nums))


---
## 8. Algorithmic Pattern 3: Prefix Sum Array

Precomputes cumulative sums so any range sum query `sum(arr[L..R])` can be answered in **$O(1)$ time**:
$$\text{RangeSum}(L, R) = \text{Prefix}[R+1] - \text{Prefix}[L]$$


In [ ]:
class PrefixSum:
    def __init__(self, arr):
        self.prefix = [0] * (len(arr) + 1)
        for i in range(len(arr)):
            self.prefix[i + 1] = self.prefix[i] + arr[i]

    def range_sum(self, left, right):
        return self.prefix[right + 1] - self.prefix[left]

data = [3, 2, -1, 6, 5, 4]
ps = PrefixSum(data)
print("Array:", data)
print("Sum of indices [1..4] (2 + -1 + 6 + 5):", ps.range_sum(1, 4))


---
## 9. Algorithmic Pattern 4: Kadane's Algorithm (Max Subarray Sum)

Finds the maximum contiguous subarray sum in $O(n)$ time and $O(1)$ space using Dynamic Programming.


In [ ]:
def max_sub_array_kadane(nums):
    max_so_far = nums[0]
    current_max = nums[0]
    
    for x in nums[1:]:
        current_max = max(x, current_max + x)
        max_so_far = max(max_so_far, current_max)
        
    return max_so_far

nums = [-2, 1, -3, 4, -1, 2, 1, -5, 4]
print("Array:", nums)
print("Maximum Subarray Sum (Kadane's):", max_sub_array_kadane(nums))


---
## 10. Quick Reference Card


In [ ]:
# ==================================================================
# ARRAYS – QUICK REFERENCE
# ==================================================================

# 2D Matrix Allocation:
matrix = [[0]*4 for _ in range(3)]

# In-place Reverse:
arr = [1, 2, 3]; arr.reverse()

# Fast Range Sum:
import itertools
pref = [0] + list(itertools.accumulate([1, 2, 3, 4]))
print("Prefix sums:", pref)


---
## Summary

| Technique / Structure | Complexity | Primary Application |
|-----------------------|------------|---------------------|
| **Index Lookup** | $O(1)$ | Random element access |
| **`array.array`** | $O(1)$ | Memory-constrained C-typed numerical storage |
| **Two Pointers** | $O(n)$ time, $O(1)$ space | Sorted arrays, reversal, pair searching |
| **Sliding Window** | $O(n)$ time, $O(1)$ space | Subarray stats over contiguous windows |
| **Prefix Sum** | $O(1)$ query, $O(n)$ prep | Fast repeated range-sum queries |
| **Kadane's Algorithm** | $O(n)$ time, $O(1)$ space | Maximum contiguous subarray sum |

---
*Next up: **Linked Lists***
